# E2.6 · Incident and disclosure obligations

**Function E — AI Governance for Agentic Systems → The Regulatory & Compliance Lead**  ·  *Security of AI*

Builds on **[E2.5 · Privacy and data protection](https://spbreed.github.io/cyber-commons/lessons/E2.5.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The question "is this reportable" has to be answerable in hours, by someone who is already busy. Trigger criteria written during an incident are written under the worst conditions available.

> **At CyberTravels.** Is the $5,000 refund incident reportable, to whom, and by when? That question gets asked at 2am by someone already busy.

## 2 · The framework

```
   the question, asked at 2am, by someone already busy

   is it reportable?  --> to whom?  --> by when?

   +---------------------------------------------+
   | trigger criteria, written in advance:       |
   | data class . autonomy . harm . jurisdiction |
   +---------------------------------------------+

   criteria written during an incident are written badly
```

Incident and disclosure obligations meet agentic incidents badly, for one
specific reason: **broken attribution consumes the clock.**

The clock starts at *awareness* — when you know a reportable event may have
occurred. It does not pause while you work out who did it. So if your logs
attribute an agent's actions to the human whose credential it borrowed (D2.1),
the days you spend establishing what actually happened are deadline days.

Two consequences worth internalising:

1. **Containing fast does not buy reporting time.** You can contain in an hour
   and still miss a 72-hour deadline.
2. **You will have to disclose before attribution is complete.** So the sentence
   you send when you know an agent acted but cannot yet say which one needs to
   be drafted *now*, not during the incident.

## 3 · Demo — the clock, and what attribution costs it

In [ ]:
import time
t0 = time.time(); H = 3600

def clock(awareness, containment, report, deadline_hours):
    to_contain = (containment - awareness)/H
    to_report  = (report - awareness)/H
    return {"contain_h": round(to_contain,1), "report_h": round(to_report,1),
            "deadline": deadline_hours, "met": to_report <= deadline_hours,
            "margin_h": round(deadline_hours - to_report, 1)}

SCENARIOS = {
 "attribution sound":            (t0 + 2*H,  t0 + 20*H),
 "attribution broken, 3d scope": (t0 + 6*H,  t0 + 92*H),
 "fast containment, slow scope": (t0 + 1*H,  t0 + 80*H),
}
print(f"{'scenario':32s}{'contain':>9}{'report':>9}{'met':>6}{'margin':>9}")
print("-" * 66)
for name, (c, r) in SCENARIOS.items():
    k = clock(t0, c, r, 72)
    print(f"{name:32s}{k['contain_h']:>9.1f}{k['report_h']:>9.1f}"
          f"{str(k['met']):>6}{k['margin_h']:>9.1f}")
print("\nThe third row contained in ONE HOUR and missed by 8 hours.")

In [ ]:
# Where the time actually goes when attribution is broken.
PHASES = [
 ("alert fires → analyst picks it up",        3,  "queue depth"),
 ("confirm an incident",                      6,  "is this real?"),
 ("establish WHO acted",                     48,  "logs name the human; agents hidden"),
 ("scope what was touched",                  24,  "must walk the delegation chain (D2.3)"),
 ("legal determines reportability",            8,  "needs the scope"),
 ("draft and send",                            3,  ""),
]
elapsed = 0
print(f"{'phase':38s}{'hours':>7}{'cumulative':>12}  note")
print("-" * 82)
for name, h, note in PHASES:
    elapsed += h
    flag = "  ← DEADLINE PASSED" if elapsed > 72 else ""
    print(f"{name:38s}{h:>7}{elapsed:>12}{flag}  {note}")
print(f"\ntotal {elapsed}h against a 72h deadline")
attribution_cost = PHASES[2][1]
print(f"the attribution phase alone is {attribution_cost}h — "
      f"{attribution_cost/72:.0%} of the entire deadline")
assert elapsed > 72

## 4 · The control — fix attribution, and pre-draft the hard sentence

In [ ]:
def with_act_chains(phases):
    """With an acting-identity field, 'who acted' is a query, not an investigation."""
    return [(n, (0.5 if n.startswith("establish WHO") else h), note)
            for n, h, note in phases]

fixed = with_act_chains(PHASES)
total_fixed = sum(h for _, h, _ in fixed)
print(f"with act chains recorded (A2.5 + EV-1): {total_fixed}h vs {elapsed}h")
print(f"deadline met: {total_fixed <= 72}")
assert total_fixed <= 72

PRE_DRAFTED = """
We are notifying you of an incident under [instrument], first identified at
[awareness timestamp].

An automated system operating within our environment performed actions that may
have affected [scope]. Our logging currently attributes these actions to the
authenticated principal on whose behalf the system was acting; we are working to
establish which specific automated component performed them.

Containment: [action] completed at [time].
We will provide an update within [period], including the completed attribution.
"""
print("\nPRE-DRAFTED DISCLOSURE (write this now, not during the incident):")
print(PRE_DRAFTED)
print("It is honest, it starts the notification, and it does not claim an")
print("attribution you cannot yet support.")

In [ ]:
# Verify: the runbook needs two owners, not one.
def runbook_check(containment_owner, disclosure_owner, clock_starts_at,
                  has_predrafted):
    problems = []
    if containment_owner == disclosure_owner:
        problems.append("one owner for both workstreams — they compete under time pressure")
    if clock_starts_at != "awareness":
        problems.append(f"clock starts at {clock_starts_at!r}; a regulator will use awareness")
    if not has_predrafted:
        problems.append("no pre-drafted disclosure for incomplete attribution")
    return (not problems), problems

for label, args in (("as usually written", ("IR lead", "IR lead", "confirmation", False)),
                    ("corrected", ("IR lead", "legal/compliance lead", "awareness", True))):
    ok, problems = runbook_check(*args)
    print(f"{label:22s} sound={ok}")
    for p in problems: print(f"   ⚠ {p}")
assert runbook_check("IR lead", "legal/compliance lead", "awareness", True)[0]

## What you just proved

One-hour containment still misses the 72-hour deadline when scoping is slow. The phase breakdown totals 92 hours, of which establishing who acted is 48 — two-thirds of the entire deadline. Recording act chains cuts the total to 44.5 hours and meets the deadline. The runbook check flags a shared owner, a late clock start and a missing pre-drafted disclosure.

## Your turn

Draft the disclosure sentence you would send when you know an agent acted but cannot say which one. Getting legal to agree that wording takes weeks in peacetime and is impossible at hour 60.

---

**Next → [E2.7 · Documentation that survives supervision](https://spbreed.github.io/cyber-commons/lessons/E2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*